# 29: How To Fail Correctly

This notebook is not another Digital Crystal experiment. It is an executable epistemic audit: when an experiment fails or narrows, can the record preserve exactly what failed, what survived, and what cannot be inferred?

In [1]:
from pathlib import Path
import json
import math
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_PROFILE = os.environ.get("NOTEBOOK_PROFILE", "quick")
RUN_CANONICAL = os.environ.get("RUN_CANONICAL", "0") == "1"


def find_repo_root(start=Path.cwd()):
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "content" / "books" / "digital-life").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate repository root")

REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
FIG_DIR = NOTEBOOK_DIR / "generated-figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def load_json(relative_path):
    path = REPO_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def require_path(relative_path):
    path = REPO_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def summarize_result(label, result, status=None, source="canonical research artifact"):
    row = {"label": label, "source": source}
    if status is not None:
        row["status"] = status
    for key in ["n", "mean", "ci95_low", "ci95_high", "achieved_mde80_one_sided"]:
        if key in result:
            row[key] = result[key]
    return row

print("profile", NOTEBOOK_PROFILE, "run_canonical", RUN_CANONICAL)
print("repo", REPO_ROOT)

CHAPTER = 29
MANUSCRIPT = require_path("content/books/digital-life/29-how-to-fail-correctly/index.md")
LINEAGE = ["scripts/books/digital-life/ch29_how_to_fail_correctly_v1.py"]
for item in LINEAGE:
    require_path(item)
print("manuscript", MANUSCRIPT.relative_to(REPO_ROOT))
print("lineage ok", len(LINEAGE))

profile quick run_canonical False
repo C:\Projects\working-book
manuscript content\books\digital-life\29-how-to-fail-correctly\index.md
lineage ok 1


In [2]:
artifact_audit = load_json("research/digital-life/ch29-how-to-fail-correctly-v1/ch29-artifact-audit.json")
ledger_json = load_json("research/digital-life/ch29-how-to-fail-correctly-v1/ch29-failure-ledger.json")
consistency = load_json("research/digital-life/ch29-how-to-fail-correctly-v1/stage-03-consistency.json")
verdict = load_json("research/digital-life/ch29-how-to-fail-correctly-v1/stage-05-verdict.json")
ledger = pd.DataFrame(ledger_json["rows"])
ledger[["case_id", "chapter", "validity", "inferential_status", "transition_type", "evidence_role"]]

,case_id,chapter,validity,inferential_status,transition_type,evidence_role
0,CH26_V1_PRIMARY,26,INVALID_REFERENCE,INVALID,IMPLEMENTATION_INVALIDATION,UNINFORMATIVE
1,CH26_V2_PRIMARY,26,VALID,BOUNDED_NEAR_ZERO,PRECISION_RESOLUTION,REFUTES
2,CH26_V2_MECHANISM,26,VALID,DESCRIPTIVE_ONLY,MECHANISTIC_DECOMPOSITION,PRESERVES_SUBRESULT
3,CH27_V1_PRIMARY,27,INVALID_IMPLEMENTATION,INVALID,IMPLEMENTATION_INVALIDATION,UNINFORMATIVE
4,CH27_V1_IMMEDIATE,27,VALID,SUPPORTED,MECHANISTIC_DECOMPOSITION,PRESERVES_SUBRESULT
5,CH27_V2_DIRECTION,27,VALID,DIRECTION_SUPPORTED,REPLICATION,SUPPORTS
6,CH27_V2_MAGNITUDE,27,VALID,UNRESOLVED,PRECISION_LIMIT,UNINFORMATIVE
7,CH27_V2_CLOSEOUT,27,VALID,DESCRIPTIVE_ONLY,DESCRIPTIVE_CLOSEOUT,PRESERVES_SUBRESULT
8,CH28_V1_RAW,28,VALID,SUPPORTED,REPLICATION,SUPPORTS
9,CH28_V2_EXCESS,28,VALID,BOUNDED_BELOW_SEI,CONTROL_STRENGTHENING,NARROWS_CLAIM


## Executable Ledger Rules

These checks are recomputed from the canonical ledger table. They verify bookkeeping consistency; they do not discover a new Crystal property.

In [3]:
checks = []
checks.append({"rule": "invalid cases are not counted as negative evidence", "pass": not ((ledger["inferential_status"].eq("INVALID")) & (ledger["evidence_role"].str.contains("NEGATIVE", na=False))).any()})
checks.append({"rule": "every registered case has a surviving evidence statement", "pass": ledger["surviving_evidence"].notna().all()})
checks.append({"rule": "confirmatory cases identify a transition type", "pass": ledger.loc[ledger["confirmatory"].eq(True), "transition_type"].notna().all()})
checks.append({"rule": "bounded cases expose thresholds or visible bounds when applicable", "pass": len(ledger[ledger["inferential_status"].str.contains("BOUNDED", na=False)]) >= 1})
local_checks = pd.DataFrame(checks)
assert local_checks["pass"].all()
local_checks

,rule,pass
0,invalid cases are not counted as negative evid...,True
1,every registered case has a surviving evidence...,True
2,confirmatory cases identify a transition type,True
3,bounded cases expose thresholds or visible bou...,True


In [4]:
status_counts = ledger.groupby(["chapter", "inferential_status"]).size().reset_index(name="cases")
ax = status_counts.pivot(index="chapter", columns="inferential_status", values="cases").fillna(0).plot(kind="bar", stacked=True, figsize=(8, 3.6))
ax.set_title("CANONICAL LEDGER: status distribution by chapter")
ax.set_ylabel("registered cases")
ax.legend(frameon=False, fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
path = FIG_DIR / "ch29-failure-ledger-status-matrix.png"
ax.figure.tight_layout(); ax.figure.savefig(path, dpi=160); plt.close(ax.figure)
path.relative_to(REPO_ROOT)

WindowsPath('notebooks/generated-figures/ch29-failure-ledger-status-matrix.png')

In [5]:
pd.DataFrame([{
    "case_rule_pass_count": consistency["case_rule_pass_count"],
    "case_rule_total": consistency["case_rule_total"],
    "all_case_rules_pass": consistency["all_case_rules_pass"],
    "cross_check_pass_count": sum(c["pass"] for c in consistency["cross_checks"]),
    "cross_check_total": len(consistency["cross_checks"]),
    "all_cross_checks_pass": consistency["all_cross_checks_pass"],
    "final_status": verdict["status"],
    "source": "canonical research artifact",
}])

,case_rule_pass_count,case_rule_total,all_case_rules_pass,cross_check_pass_count,cross_check_total,all_cross_checks_pass,final_status,source
0,10,10,True,5,5,True,FAILURE_LEDGER_CONSISTENT,canonical research artifact


## Didactic Wrong Moves

The rows below are synthetic examples. They are deliberately not project evidence.

In [6]:
wrong = pd.DataFrame([
    {"wrong_move": "INVALID -> hypothesis false", "why_wrong": "invalid implementation is uninformative about the intended estimand"},
    {"wrong_move": "CI crosses zero -> FAILED", "why_wrong": "unresolved and bounded statuses require thresholds and precision"},
    {"wrong_move": "descriptive closeout -> confirmatory rescue", "why_wrong": "post-hoc trajectory description cannot satisfy the frozen primary gate"},
])
wrong

,wrong_move,why_wrong
0,INVALID -> hypothesis false,invalid implementation is uninformative about ...
1,CI crosses zero -> FAILED,unresolved and bounded statuses require thresh...
2,descriptive closeout -> confirmatory rescue,post-hoc trajectory description cannot satisfy...


## Result

The canonical audit reports `FAILURE_LEDGER_CONSISTENT`: 10 / 10 case checks pass and 5 / 5 cross-case checks pass. This is an epistemic-integrity result, not a biological property claim.